#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Analysis

## Raw Data

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

df = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time.csv')

if not df.empty:
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()

    cat_mapping = {
        'keywords_General_Dentist': 'General',
        'keywords_Special_Dentist': 'Special',
        'keywords_Surgery_Dentist': 'Surgery'
    }

    if 'category' in df.columns:
        df = df[df['category'].isin(cat_mapping.keys())].copy()
        df['Keyword_Category'] = df['category'].map(cat_mapping)
    else:
        df['Keyword_Category'] = 'Unknown'

    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'],
        ordered=False
    )

    df['Category_Location'] = df['Keyword_Category'].astype(str) + '_' + df['Location_Type'].astype(str)

    df['Category_Location'] = pd.Categorical(
        df['Category_Location'],
        categories=[
            'General_Mid_Size',
            'General_Large', 'General_Small',
            'Special_Mid_Size', 'Special_Large', 'Special_Small',
            'Surgery_Mid_Size', 'Surgery_Large', 'Surgery_Small'
        ],
        ordered=False
    )

    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 3958.8
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_to_hub'] = df.apply(get_distance, axis=1)

    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    def calc_peer_dist(sub):
        if len(sub) < 2: return np.zeros(len(sub))
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dists = c * 3958.8
        return np.sum(dists, axis=1) / (len(sub) - 1)

    coords = df[['latitude', 'longitude', 'zip_clean']].dropna()
    peer_dist_map = {}
    for z, group in coords.groupby('zip_clean'):
        if len(group) > 1:
            dists = calc_peer_dist(group)
            for idx, val in zip(group.index, dists):
                peer_dist_map[idx] = val
        else:
            for idx in group.index:
                peer_dist_map[idx] = 0.0

    df['avg_peer_dist'] = df.index.map(peer_dist_map).fillna(0)
    df['avg_peer_dist'] = df['avg_peer_dist'].replace(0.0, np.nan)

    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')

    df_low = df[df['rating_value'] <= 3.0].copy()
    peer_dist_low_map = {}
    if not df_low.empty:
        coords_low = df_low[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_low.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_low_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_low_map[idx] = 0.0

    df['avg_dist_to_1star_peers'] = df.index.map(peer_dist_low_map)
    df['avg_dist_to_1star_peers'] = df['avg_dist_to_1star_peers'].replace(0.0, np.nan)

    df_other = df[df['rating_value'] > 3.0].copy()
    peer_dist_other_map = {}
    if not df_other.empty:
        coords_other = df_other[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_other.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_other_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_other_map[idx] = 0.0

    df['avg_dist_to_other_peers'] = df.index.map(peer_dist_other_map)
    df['avg_dist_to_other_peers'] = df['avg_dist_to_other_peers'].replace(0.0, np.nan)

    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))
    df['is_low_rating'] = np.where(df['rating_value'] <= 3, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=[
        'rating_value', 'distance_to_hub', 'log_votes',
        'zip_clinic_count', 'State', 'Category_Location', 'Location_Type'
    ]).copy()

    print(f"N = {len(reg_df)}")

## Panel

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
from sklearn.neighbors import BallTree
from sklearn.metrics import DistanceMetric
import warnings
warnings.filterwarnings('ignore')

print("Build Panel Data")

df_reg = df.dropna(subset=['rating_value', 'latitude', 'longitude', 'start_date', 'mapped_location']).copy()
df_reg['entry_year'] = pd.to_datetime(df_reg['start_date'], errors='coerce').dt.year
df_reg = df_reg.dropna(subset=['entry_year'])
df_reg['entry_year'] = df_reg['entry_year'].astype(int)

df_reg = df_reg.reset_index(drop=True)
df_reg['clinic_id'] = ["C_" + str(i) for i in range(len(df_reg))]

# strong competitor
df_reg['city_avg_votes'] = df_reg.groupby('mapped_location')['votes_count'].transform('mean')
df_reg['is_strong'] = (df_reg['rating_value'] > 4.9) & (df_reg['votes_count'] > df_reg['city_avg_votes'])

# 25% and 50% of the distance between clinics within each city
EARTH_RADIUS = 3958.8
dist = DistanceMetric.get_metric('haversine')
city_thresholds = {}

for city, group in df_reg.groupby('mapped_location'):
    coords = np.radians(group[['latitude', 'longitude']].values)
    if len(coords) > 1:
        dist_matrix = dist.pairwise(coords) * EARTH_RADIUS
        triu_indices = np.triu_indices(len(coords), k=1)
        pairwise_dists = dist_matrix[triu_indices]
        p25 = np.percentile(pairwise_dists, 25) / EARTH_RADIUS
        p50 = np.percentile(pairwise_dists, 50) / EARTH_RADIUS
    else:
        p25, p50 = 0, 0
    city_thresholds[city] = {'p25': p25, 'p50': p50}

coords_rad = np.radians(df_reg[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')

# For each clinic get the neighbors within 25% and 50% radius.
indices_25pct = []
indices_50pct = []
for i, row in df_reg.iterrows():
    city = row['mapped_location']
    r_25 = city_thresholds.get(city, {'p25': 0})['p25']
    r_50 = city_thresholds.get(city, {'p50': 0})['p50']

    pt = coords_rad[i:i+1]
    indices_25pct.append(tree.query_radius(pt, r=r_25)[0])
    indices_50pct.append(tree.query_radius(pt, r=r_50)[0])

panel_records = []
for i, row in df_reg.iterrows():
    entry_y = int(row['entry_year'])

    neighbors_25 = indices_25pct[i]
    years_25 = df_reg.iloc[neighbors_25]['entry_year'].values
    titles_25 = df_reg.iloc[neighbors_25]['title'].values
    ids_25 = df_reg.iloc[neighbors_25]['clinic_id'].values
    strong_25 = df_reg.iloc[neighbors_25]['is_strong'].values

    neighbors_50 = indices_50pct[i]
    years_50 = df_reg.iloc[neighbors_50]['entry_year'].values
    titles_50 = df_reg.iloc[neighbors_50]['title'].values
    ids_50 = df_reg.iloc[neighbors_50]['clinic_id'].values
    strong_50 = df_reg.iloc[neighbors_50]['is_strong'].values

    for current_year in range(entry_y, 2026):
        # 25%
        density_25 = max(0, np.sum(years_25 <= current_year) - 1)
        lag_25 = np.sum(years_25 == (current_year - 1))
        shock_mask_25 = (years_25 == (current_year - 1)) & (ids_25 != row['clinic_id'])
        strong_lag_25 = np.sum((years_25 == (current_year - 1)) & strong_25 & (ids_25 != row['clinic_id']))
        names_25 = ", ".join(titles_25[shock_mask_25])

        # 50%
        density_50 = max(0, np.sum(years_50 <= current_year) - 1)
        lag_50 = np.sum(years_50 == (current_year - 1))
        shock_mask_50 = (years_50 == (current_year - 1)) & (ids_50 != row['clinic_id'])
        strong_lag_50 = np.sum((years_50 == (current_year - 1)) & strong_50 & (ids_50 != row['clinic_id']))
        names_50 = ", ".join(titles_50[shock_mask_50])

        # 25%-50%
        density_50_ring = max(0, np.sum(years_50 <= current_year) - 1) - density_25
        lag_50_ring = np.sum(years_50 == (current_year - 1)) - lag_25
        shock_mask_50_ring = (years_50 == (current_year - 1)) & (ids_50 != row['clinic_id']) & ~np.isin(ids_50, ids_25)
        strong_lag_50_ring = strong_lag_50 - strong_lag_25
        names_50_ring = ", ".join(titles_50[shock_mask_50_ring])

        panel_records.append({
            'clinic_id': row['clinic_id'],
            'title': row['title'],
            'zip_clean': row['zip_clean'],
            'year': current_year,
            'clinic_age': current_year - entry_y,

            # 25%
            'density_25pct_total': density_25,
            'log_density_25pct_total': np.log1p(density_25),
            'lag_entry_shock_25pct_count': lag_25,
            'lag_entry_shock_25pct_names': names_25,
            'lag_entry_shock_25pct_dummy': 1 if lag_25 > 0 else 0,
            'log_lag_entry_shock_25pct': np.log1p(lag_25),
            'strong_entry_shock_25pct_count': strong_lag_25,
            'log_strong_entry_shock_25pct': np.log1p(strong_lag_25),

            # 50%
            'density_50pct_total': density_50,
            'log_density_50pct_total': np.log1p(density_50),
            'lag_entry_shock_50pct_count': lag_50,
            'lag_entry_shock_50pct_names': names_50,
            'lag_entry_shock_50pct_dummy': 1 if lag_50 > 0 else 0,
            'log_lag_entry_shock_50pct': np.log1p(lag_50),
            'strong_entry_shock_50pct_count': strong_lag_50,
            'log_strong_entry_shock_50pct': np.log1p(strong_lag_50),

            # 25%-50%
            'density_25_to_50pct_total': density_50_ring,
            'log_density_25_to_50pct_total': np.log1p(density_50_ring),
            'lag_entry_shock_25_to_50pct_count': lag_50_ring,
            'lag_entry_shock_25_to_50pct_names': names_50_ring,
            'lag_entry_shock_25_to_50pct_dummy': 1 if lag_50_ring > 0 else 0,
            'log_lag_entry_shock_25_to_50pct': np.log1p(lag_50_ring),
            'strong_entry_shock_25_to_50pct_count': strong_lag_50_ring,
            'log_strong_entry_shock_25_to_50pct': np.log1p(strong_lag_50_ring),

            'distance_to_hub': row['distance_to_hub'],
            'mapped_location': row['mapped_location'],
            'Keyword_Category': row['Keyword_Category']
        })

df_panel = pd.DataFrame(panel_records)

# Process all review for dynamic ratings and votes
REVIEW_NAME_COL = 'shop_title'
REVIEW_ZIP_COL = 'shop_zip'
REVIEW_DATE_COL = 'date'
REVIEW_RATING_COL = 'rating'

chunk_size = 10000
yearly_stats_chunks = []
cols_to_use = [REVIEW_NAME_COL, REVIEW_ZIP_COL, REVIEW_DATE_COL, REVIEW_RATING_COL]

print("Chunking and aggregating reviews...")
try:
    for chunk in pd.read_csv("/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv", chunksize=chunk_size, usecols=cols_to_use):
        # match zip code
        chunk[REVIEW_ZIP_COL] = chunk[REVIEW_ZIP_COL].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        chunk['review_year'] = pd.to_datetime(chunk[REVIEW_DATE_COL], errors='coerce').dt.year
        chunk = chunk.dropna(subset=['review_year', REVIEW_RATING_COL])
        chunk['review_year'] = chunk['review_year'].astype(int)

        chunk['is_bad_review'] = (chunk[REVIEW_RATING_COL] <= 3).astype(int)

        # aggregate count and sum per year
        stats_df = chunk.groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year']).agg(
            new_count=(REVIEW_RATING_COL, 'count'),
            new_stars=(REVIEW_RATING_COL, 'sum'),
            bad_count=('is_bad_review', 'sum')
        ).reset_index()
        yearly_stats_chunks.append(stats_df)

    final_yearly_stats = pd.concat(yearly_stats_chunks).groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year']).sum().reset_index()
    final_yearly_stats = final_yearly_stats.rename(columns={REVIEW_NAME_COL: 'title', REVIEW_ZIP_COL: 'zip_str'})
except Exception as e:
    print(f"   [Warning] Failed to read reviews: {e}")

# Merge reviews into panel
df_panel['zip_str'] = df_panel['zip_clean'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_panel = df_panel.merge(final_yearly_stats, left_on=['title', 'zip_str', 'year'], right_on=['title', 'zip_str', 'review_year'], how='left')
df_panel['new_count'] = df_panel['new_count'].fillna(0)
df_panel['new_stars'] = df_panel['new_stars'].fillna(0)
df_panel['bad_count'] = df_panel['bad_count'].fillna(0)

# sort
df_panel = df_panel.sort_values(by=['clinic_id', 'year'])
df_panel['cumulative_votes'] = df_panel.groupby('clinic_id')['new_count'].cumsum()
df_panel['cumulative_stars'] = df_panel.groupby('clinic_id')['new_stars'].cumsum()
df_panel['cumulative_bad_votes'] = df_panel.groupby('clinic_id')['bad_count'].cumsum()

# dynamic vote
df_panel['dynamic_rating'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_stars'] / df_panel['cumulative_votes'], np.nan)
df_panel['log_votes_dynamic'] = np.log1p(df_panel['cumulative_votes'])


df_panel['current_bad_review_pct'] = np.where(df_panel['new_count'] > 0, df_panel['bad_count'] / df_panel['new_count'], np.nan)


df_panel['cumulative_bad_review_pct'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_bad_votes'] / df_panel['cumulative_votes'], np.nan)


# drop early years where the clinic hadn't received first review
df_panel_valid = df_panel.dropna(subset=['dynamic_rating']).reset_index(drop=True)

In [ ]:
!pip install linearmodels

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import numpy as np

df_panel_valid['other_entry_shock_25pct_count'] = df_panel_valid['lag_entry_shock_25pct_count'] - df_panel_valid['strong_entry_shock_25pct_count']
df_panel_valid['log_other_entry_shock_25pct'] = np.log1p(df_panel_valid['other_entry_shock_25pct_count'])

df_panel_valid['other_entry_shock_25_to_50pct_count'] = df_panel_valid['lag_entry_shock_25_to_50pct_count'] - df_panel_valid['strong_entry_shock_25_to_50pct_count']
df_panel_valid['log_other_entry_shock_25_to_50pct'] = np.log1p(df_panel_valid['other_entry_shock_25_to_50pct_count'])

df_panel_valid['other_entry_shock_50pct_count'] = df_panel_valid['lag_entry_shock_50pct_count'] - df_panel_valid['strong_entry_shock_50pct_count']
df_panel_valid['log_other_entry_shock_50pct'] = np.log1p(df_panel_valid['other_entry_shock_50pct_count'])





In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

shock_configs = [
    ('strong_entry_shock_25pct_count', '0-25%'),
    ('strong_entry_shock_25_to_50pct_count', '25-50%'),
    ('strong_entry_shock_50pct_count', '0-50%')
]

sns.set_theme(style="white")
palette = {'Before Strong Entry': '#3498db', 'After Strong Entry': '#e74c3c'}

for shock_col, label in shock_configs:
    if isinstance(df_panel_valid.index, pd.MultiIndex):
        df_plot_data = df_panel_valid.reset_index()
    else:
        df_plot_data = df_panel_valid.copy()

    df_plot_data['post_strong_entry'] = (df_plot_data.groupby('clinic_id')[shock_col].cumsum() > 0).astype(int)

    clinics_with_shock = df_plot_data[df_plot_data['post_strong_entry'] == 1]['clinic_id'].unique()
    df_plot_data = df_plot_data[df_plot_data['clinic_id'].isin(clinics_with_shock)].copy()

    if df_plot_data.empty:
        continue

    df_plot_data['Comp_Group'] = df_plot_data['post_strong_entry'].map({
        0: 'Before Strong Entry',
        1: 'After Strong Entry'
    })

    mean_before = df_plot_data[df_plot_data['Comp_Group'] == 'Before Strong Entry']['dynamic_rating'].mean()
    mean_after = df_plot_data[df_plot_data['Comp_Group'] == 'After Strong Entry']['dynamic_rating'].mean()

    fig_kde, ax_kde = plt.subplots(figsize=(10, 6), dpi=100)
    sns.kdeplot(data=df_plot_data, x="dynamic_rating", hue="Comp_Group",
                fill=True, alpha=0.5, bw_adjust=1.2, palette=palette,
                lw=2, ax=ax_kde, common_norm=False)

    ax_kde.axvline(mean_before, color='#2980b9', linestyle='--', lw=2)
    ax_kde.text(mean_before*0.98, ax_kde.get_ylim()[1]*0.85, f'Mean Before: {mean_before:.2f}',
                fontsize=11, fontweight='bold', color='#2980b9', ha='right')

    ax_kde.axvline(mean_after, color='#c0392b', linestyle='--', lw=2)
    ax_kde.text(mean_after*1.02, ax_kde.get_ylim()[1]*0.75, f'Mean After: {mean_after:.2f}',
                fontsize=11, fontweight='bold', color='#c0392b', ha='left')

    ax_kde.axvline(3.5, color='black', linestyle='-', lw=2)
    ax_kde.text(3.5*0.98, ax_kde.get_ylim()[1]*0.5, 'Cutoff: 3.5',
                fontsize=11, fontweight='bold', color='black', ha='right')

    ax_kde.set_xlim(2.5, 5.1)
    ax_kde.set_xlabel("Dynamic Rating", fontsize=12)
    ax_kde.set_ylabel("Density", fontsize=12)
    ax_kde.set_title(f"Density Shift: Before vs. After Strong Entry ({label})", fontsize=15, fontweight='bold', y=1.05)
    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

    fig_hist, ax_hist = plt.subplots(figsize=(10, 6), dpi=100)
    sns.histplot(data=df_plot_data, x="dynamic_rating", hue="Comp_Group",
                 palette=palette, edgecolor='white', bins=15, alpha=0.6,
                 stat='density', ax=ax_hist, common_norm=False, multiple="layer")

    ax_hist.axvline(mean_before, color='#2980b9', linestyle='--', lw=2)
    ax_hist.text(mean_before*0.98, ax_hist.get_ylim()[1]*0.85, f'Mean Before: {mean_before:.2f}',
                 fontsize=11, fontweight='bold', color='#2980b9', ha='right')

    ax_hist.axvline(mean_after, color='#c0392b', linestyle='--', lw=2)
    ax_hist.text(mean_after*1.02, ax_hist.get_ylim()[1]*0.75, f'Mean After: {mean_after:.2f}',
                 fontsize=11, fontweight='bold', color='#c0392b', ha='left')

    ax_hist.axvline(3.5, color='black', linestyle='-', lw=2)

    ax_hist.set_xlim(2.5, 5.1)
    ax_hist.set_xlabel("Dynamic Rating", fontsize=12)
    ax_hist.set_ylabel("Density", fontsize=12)
    ax_hist.set_title(f"Histogram Shift: Before vs. After Strong Entry ({label})", fontsize=15, fontweight='bold', y=1.05)
    sns.despine(left=True)
    plt.tight_layout()
    plt.show()

In [ ]:
shock_configs = [
    ('strong_entry_shock_25pct_count', '0-25%'),
    ('strong_entry_shock_25_to_50pct_count', '25-50%'),
    ('strong_entry_shock_50pct_count', '0-50%')
]

bins = [1.0, 2.0, 3.0, 4.0, 5.0]
labels = ["1-2", "2-3",  "3-4",  "4-5"]

for shock_col, label in shock_configs:
    print(f"\nShock: {label} ({shock_col})")

    if isinstance(df_panel_valid.index, pd.MultiIndex):
        df_temp = df_panel_valid.reset_index()
    else:
        df_temp = df_panel_valid.copy()

    df_temp['post_strong_entry'] = (df_temp.groupby('clinic_id')[shock_col].cumsum() > 0).astype(int)

    clinics_with_shock = df_temp[df_temp['post_strong_entry'] == 1]['clinic_id'].unique()
    df_temp = df_temp[df_temp['clinic_id'].isin(clinics_with_shock)].copy()

    if df_temp.empty:
        print("No data.")
        continue

    df_temp['Comp_Group'] = df_temp['post_strong_entry'].map({
        0: 'Before Strong Entry',
        1: 'After Strong Entry'
    })

    for group in df_temp['Comp_Group'].unique():
        group_data = df_temp[df_temp['Comp_Group'] == group]['dynamic_rating']
        count = pd.cut(group_data, bins=bins, labels=labels).value_counts()
        percent = (count / len(group_data)) * 100
        for interval, percentage in percent.sort_index().items():
            print(f"{group} {interval}: {percentage:.2f}%")


#LLM

In [ ]:
!pip install transformers torch tqdm pandas

In [ ]:
!pip install linearmodels

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm


device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=device)


In [ ]:
df_reviews = pd.read_csv("/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed.csv")
df_reviews = df_reviews.dropna(subset=['review_text', 'timestamp', 'shop_title'])
df_reviews['review_year'] = pd.to_datetime(df_reviews['timestamp']).dt.year
df_reviews['review_text'] = df_reviews['review_text'].astype(str)

In [ ]:
INTERMEDIATE_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/NLP/nlp_intermediate_results_clean.csv"
FINAL_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits_clean.csv"

os.makedirs(os.path.dirname(INTERMEDIATE_FILE), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_FILE), exist_ok=True)

print("\nRegex for Medical Services")
#bait
negation_prefix = r'\b(no|not|without|avoid|avoided|never)\s+(need|needed|get|got|have|having)?\s*(\w+\s+)?'

ortho_regex = r'\b(brace|braces|invisalign|ortho|orthodontic|orthodontist|retainer)\b'
ortho_bait = negation_prefix + r'(brace|braces|invisalign|ortho|orthodontic|orthodontist|retainer)\b'
df_reviews['service_ortho'] = (
    df_reviews['review_text'].str.contains(ortho_regex, case=False, na=False, regex=True) &
    ~df_reviews['review_text'].str.contains(ortho_bait, case=False, na=False, regex=True)
).astype(int)

cosmetic_regex = r'\b(whitening|veneer|veneers|cosmetic|bleaching)\b'
cosmetic_bait = negation_prefix + r'(whitening|veneer|veneers|cosmetic|bleaching)\b'
df_reviews['service_cosmetic'] = (
    df_reviews['review_text'].str.contains(cosmetic_regex, case=False, na=False, regex=True) &
    ~df_reviews['review_text'].str.contains(cosmetic_bait, case=False, na=False, regex=True)
).astype(int)

surgery_regex = r'\b(surgery|implant|implants|extraction|extracted|pull|pulled|wisdom)\b'
surgery_bait = negation_prefix + r'(surgery|implant|implants|extraction|extracted|pull|pulled|wisdom)\b'
df_reviews['service_surgery'] = (
    df_reviews['review_text'].str.contains(surgery_regex, case=False, na=False, regex=True) &
    ~df_reviews['review_text'].str.contains(surgery_bait, case=False, na=False, regex=True)
).astype(int)

# LLM + Bait
target_labels = [
    "advertisement, coupon, promotion, or special discount",
    "pediatric dentistry, kids, or elderly care",
    "this clinic is too expensive, high price, overcharged, or hidden fees",
    "long wait time, rude staff, or bad customer service",
    "comparing to another dentist or previous clinic"
]

bait_labels = [
    "highly recommended by patient, great experience, positive review",
    "general family dentistry, referring my family and friends",
    "affordable, fair price, cheap, or great value",
    "the previous dentist was expensive, other clinic overcharged",
    "painful procedure, bad medical outcome, or dentist made a mistake",
    "the best or worst experience ever, general great experience"
]

candidate_labels = target_labels + bait_labels
THRESHOLD = 0.6
batch_size = 32
CHUNK_SIZE = 10000

if os.path.exists(INTERMEDIATE_FILE):
    processed_df = pd.read_csv(INTERMEDIATE_FILE, usecols=['shop_title'])
    start_idx = len(processed_df)
    print(f"\nFound existing records. Resuming from row {start_idx} / {len(df_reviews)}...")
else:
    start_idx = 0
    print(f"\nTotal rows to process: {len(df_reviews)}")

for chunk_start in range(start_idx, len(df_reviews), CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(df_reviews))
    print(f"\n>>> Processing rows {chunk_start} to {chunk_end}...")

    chunk_df = df_reviews.iloc[chunk_start:chunk_end].copy()
    texts = chunk_df['review_text'].apply(lambda x: x[:512]).tolist()

    chunk_results = []

    for out in tqdm(classifier(texts, candidate_labels, multi_label=True, batch_size=batch_size), total=len(texts)):
        scores = dict(zip(out['labels'], out['scores']))
        chunk_results.append(scores)

    # bait>0.6 or target>bait
    ad_list, demo_list, price_list, service_list, compare_list = [], [], [], [], []

    for r in chunk_results:
        tgt = r["advertisement, coupon, promotion, or special discount"]
        bait = r["highly recommended by patient, great experience, positive review"]
        ad_list.append(1 if (tgt > THRESHOLD and bait <= 0.6 and bait <= tgt) else 0)

        tgt = r["pediatric dentistry, kids, or elderly care"]
        bait = r["general family dentistry, referring my family and friends"]
        demo_list.append(1 if (tgt > THRESHOLD and bait <= 0.6 and bait <= tgt) else 0)

        tgt = r["this clinic is too expensive, high price, overcharged, or hidden fees"]
        bait_1 = r["affordable, fair price, cheap, or great value"]
        bait_2 = r["the previous dentist was expensive, other clinic overcharged"]
        bait_max = max(bait_1, bait_2)
        price_list.append(1 if (tgt > THRESHOLD and bait_max <= 0.6 and bait_max <= tgt) else 0)

        tgt = r["long wait time, rude staff, or bad customer service"]
        bait = r["painful procedure, bad medical outcome, or dentist made a mistake"]
        service_list.append(1 if (tgt > THRESHOLD and bait <= 0.6 and bait <= tgt) else 0)

        tgt = r["comparing to another dentist or previous clinic"]
        bait = r["the best or worst experience ever, general great experience"]
        compare_list.append(1 if (tgt > THRESHOLD and bait <= 0.6 and bait <= tgt) else 0)

    chunk_df['is_ad_driven'] = ad_list
    chunk_df['target_demographic'] = demo_list
    chunk_df['issue_price'] = price_list
    chunk_df['issue_service'] = service_list
    chunk_df['is_comparing'] = compare_list

    chunk_df.to_csv(INTERMEDIATE_FILE, mode='a', header=not os.path.exists(INTERMEDIATE_FILE), index=False)
    print(f"Chunk {chunk_start} to {chunk_end} saved")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from tqdm.auto import tqdm

INTERMEDIATE_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/temp/NLP/nlp_intermediate_results_clean.csv"
FINAL_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits_clean.csv"
df_full_processed = pd.read_csv(INTERMEDIATE_FILE)

print("Filtering")

# delete pet
df_full_processed = df_full_processed[~df_full_processed['review_text'].str.contains(r'\b(cat|dog|vet|veterinary|puppy|kitten|feline|canine)\b', case=False, na=False)].copy()

# Compare
strict_compare_regex = r'\b(previous dentist|other dentist|another dentist|old dentist|other clinic|another clinic|used to go|switched to|better than|worse than|another doctor|other office|new dentist)\b'
has_compare_kw = df_full_processed['review_text'].str.contains(strict_compare_regex, case=False, na=False)
df_full_processed['is_comparing'] = ((df_full_processed['is_comparing'] == 1) & has_compare_kw).astype(int)

# Demo
has_demo_kw = df_full_processed['review_text'].str.contains(r'\b(kid|child|children|pediatric|son|daughter|elderly|senior|parent)\b', case=False, na=False)
df_full_processed['target_demographic'] = ((df_full_processed['target_demographic'] == 1) & has_demo_kw).astype(int)

# ad
has_ad_kw = df_full_processed['review_text'].str.contains(r'\b(advertised|ad|coupon|promotion|special|discount|promo|groupon)\b', case=False, na=False)
df_full_processed['is_ad_driven'] = ((df_full_processed['is_ad_driven'] == 1) & has_ad_kw & (df_full_processed['rating'] > 4)).astype(int)

# Price
df_full_processed['issue_price'] = ((df_full_processed['issue_price'] == 1) & (df_full_processed['rating'] <= 3)).astype(int)

# Service
not_long_wait = df_full_processed['review_text'].str.contains(r'not a long wait', case=False, na=False)
df_full_processed['issue_service'] = ((df_full_processed['issue_service'] == 1) & (df_full_processed['rating'] <= 3) & ~not_long_wait).astype(int)

# categories
negation_prefix = r'\b(no|not|without|avoid|avoided|never|didn\'t need|don\'t need|prevent)\s+(need|needed|get|got|have|having)?\s*(\w+\s+)?'

ortho_regex = r'\b(brace|braces|invisalign|ortho|orthodontic|orthodontist|retainer)\b'
df_full_processed['service_ortho'] = (
    df_full_processed['review_text'].str.contains(ortho_regex, case=False, na=False, regex=True) &
    ~df_full_processed['review_text'].str.contains(negation_prefix + ortho_regex, case=False, na=False, regex=True)
).astype(int)

cosmetic_regex = r'\b(whitening|veneer|veneers|cosmetic|bleaching)\b'
df_full_processed['service_cosmetic'] = (
    df_full_processed['review_text'].str.contains(cosmetic_regex, case=False, na=False, regex=True) &
    ~df_full_processed['review_text'].str.contains(negation_prefix + cosmetic_regex, case=False, na=False, regex=True)
).astype(int)

surgery_regex = r'\b(surgery|implant|implants|extraction|extracted|pull|pulled|wisdom)\b'
df_full_processed['service_surgery'] = (
    df_full_processed['review_text'].str.contains(surgery_regex, case=False, na=False, regex=True) &
    ~df_full_processed['review_text'].str.contains(negation_prefix + surgery_regex, case=False, na=False, regex=True)
).astype(int)


df_full_processed['shop_zip'] = df_full_processed['shop_zip'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_clinic_traits = df_full_processed.groupby(['shop_title', 'shop_zip', 'review_year']).agg(
    total_reviews=('rating', 'count'),
    ad_ratio=('is_ad_driven', 'mean'),
    demo_ratio=('target_demographic', 'mean'),
    cosmetic_ratio=('service_cosmetic', 'mean'),
    ortho_ratio=('service_ortho', 'mean'),
    surgery_ratio=('service_surgery', 'mean'),
    price_issue_ratio=('issue_price', 'mean'),
    service_issue_ratio=('issue_service', 'mean'),
    compare_ratio=('is_comparing', 'mean')
).reset_index()

# ad expenditure proxy
df_clinic_traits['ad_expenditure_proxy'] = df_clinic_traits['total_reviews'] * df_clinic_traits['ad_ratio']
df_clinic_traits['log_ad_expenditure'] = np.log1p(df_clinic_traits['ad_expenditure_proxy'])
df_clinic_traits = df_clinic_traits.rename(columns={
    'shop_title': 'title',
    'shop_zip': 'zip_str',
    'review_year': 'year'
})

df_clinic_traits.to_csv(FINAL_FILE, index=False)
print(f"Saved to:\n{FINAL_FILE}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


print("Merging NLP results into df_panel_valid")

NLP_FILE_CLEAN = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/clinic_yearly_nlp_traits_clean.csv"
df_nlp_clean = pd.read_csv(NLP_FILE_CLEAN)

nlp_cols = [
    'ad_ratio', 'demo_ratio', 'cosmetic_ratio', 'ortho_ratio',
    'surgery_ratio', 'price_issue_ratio', 'service_issue_ratio',
    'compare_ratio', 'log_ad_expenditure'
]
df_panel_valid = df_panel_valid.drop(columns=[c for c in nlp_cols if c in df_panel_valid.columns], errors='ignore')


df_panel_valid = df_panel_valid.merge(
    df_nlp_clean,
    on=['title', 'zip_str', 'year'],
    how='left'
)


for col in nlp_cols:
    if col in df_panel_valid.columns:
        df_panel_valid[col] = df_panel_valid[col].fillna(0)

# lag
df_panel_valid = df_panel_valid.sort_values(by=['clinic_id', 'year'])
for col in nlp_cols:
    if col in df_panel_valid.columns:
        df_panel_valid[f'lag_{col}'] = df_panel_valid.groupby('clinic_id')[col].shift(1)
        df_panel_valid[f'lag_{col}'] = df_panel_valid[f'lag_{col}'].fillna(0)


PANEL_OUTPUT_PATH_CLEAN = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/panel_data_with_nlp_clean.csv"

df_panel_valid.to_csv(PANEL_OUTPUT_PATH_CLEAN, index=False)

print(f"Saved to: {PANEL_OUTPUT_PATH_CLEAN} (Rows: {len(df_panel_valid)})")

In [ ]:
PANEL_OUTPUT_PATH_CLEAN = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/panel_data_with_nlp_clean.csv"
df_panel_valid = pd.read_csv(PANEL_OUTPUT_PATH_CLEAN)
nlp_ratios = [
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio'
]


total_reviews_in_panel = df_panel_valid['new_count'].sum()

for col in nlp_ratios:
    if col in df_panel_valid.columns:
        total_mentions = (df_panel_valid[col] * df_panel_valid['new_count']).sum()
        overall_pct = (total_mentions / total_reviews_in_panel) * 100
        print(f"{col:<20} : {overall_pct:>5.2f}%")

In [ ]:
y_vars = ['dynamic_rating',
    'current_bad_review_pct',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio',
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio']

summary_stats_y = df_panel_valid[[var for var in y_vars if var in df_panel_valid.columns]].describe().T
summary_stats_y

In [ ]:
print("Random 5 Review Samples per Category (CLEANED)\n")

ratio_to_binary = {
    'ad_ratio': 'is_ad_driven',
    'demo_ratio': 'target_demographic',
    'cosmetic_ratio': 'service_cosmetic',
    'ortho_ratio': 'service_ortho',
    'surgery_ratio': 'service_surgery',
    'price_issue_ratio': 'issue_price',
    'service_issue_ratio': 'issue_service',
    'compare_ratio': 'is_comparing'
}

for ratio_col, binary_col in ratio_to_binary.items():
    print(f"[{ratio_col.upper()}] (Filter: {binary_col} == 1)")

    subset = df_full_processed[df_full_processed[binary_col] == 1]

    n_samples = min(5, len(subset))
    if n_samples > 0:
        # random_state=42
        samples = subset.sample(n=n_samples, random_state=42)['review_text'].tolist()
        for i, text in enumerate(samples, 1):
            clean_text = str(text).replace('\n', ' ').strip()
            display_text = clean_text
            print(f"  {i}. {display_text}")
    else:
        print("  No reviews found for this category.")
    print("-" * 80)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

FINAL_PANEL_FILE = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/panel_data_with_nlp_clean.csv"
df = pd.read_csv(FINAL_PANEL_FILE)

ratios_to_plot = [
    'ad_ratio',
    'demo_ratio',
    'cosmetic_ratio',
    'ortho_ratio',
    'surgery_ratio',
    'price_issue_ratio',
    'service_issue_ratio',
    'compare_ratio'
]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

ratios_to_plot = [
    'ad_ratio', 'demo_ratio', 'cosmetic_ratio', 'ortho_ratio',
    'surgery_ratio', 'price_issue_ratio', 'service_issue_ratio', 'compare_ratio'
]

df_plot = df.dropna(subset=['dynamic_rating', 'density_25pct_total'] + ratios_to_plot).copy()

# high/low competiton by density 25%
median_density = df_plot['density_25pct_total'].median()
df_plot['Competition_Level'] = df_plot['density_25pct_total'].apply(
    lambda x: 'High Competition' if x > median_density else 'Low Competition'
)

# high/low rating by density 25%
median_rating = df_plot['dynamic_rating'].median()
df_plot['Rating_Level'] = df_plot['dynamic_rating'].apply(
    lambda x: 'High Rating' if x > median_rating else 'Low Rating'
)

df_melt = df_plot.melt(
    id_vars=['Competition_Level', 'Rating_Level'],
    value_vars=ratios_to_plot,
    var_name='Text',
    value_name='Average_Ratio'
)

df_melt['Text'] = df_melt['Text'].str.replace('_ratio', '').str.replace('_', ' ').str.title()

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 12))

# N_it
sns.barplot(
    data=df_melt,
    x='Text',
    y='Average_Ratio',
    hue='Competition_Level',
    ax=axes[0],
    palette='Blues_d',
    errorbar=('ci', 95), # 95% CI
    capsize=0.1
)
axes[0].set_title('Average Ratios by Competition Level (High vs. Low $N_{it}$)', fontweight='bold')
axes[0].set_ylabel('Average Ratio')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='Density Level')

# Rating
sns.barplot(
    data=df_melt,
    x='Text',
    y='Average_Ratio',
    hue='Rating_Level',
    ax=axes[1],
    palette='Oranges_d',
    errorbar=('ci', 95),
    capsize=0.1
)
axes[1].set_title('Average Ratios by Clinic Rating (High vs. Low Rating)', fontweight='bold')
axes[1].set_ylabel('Average Ratio')
axes[1].set_xlabel('Text Dimensions extracted via NLP')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Rating Level')

plt.tight_layout()
plt.show()

# Regression

In [ ]:
# 25%
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm


# ZIP5 x Year
df_panel_valid['zip5_year'] = df_panel_valid['zip_str'].astype(str) + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['zip5_year'] = df_panel_valid['zip5_year'].astype('category')

# ZIP3 x Year
df_panel_valid['zip3'] = df_panel_valid['zip_str'].astype(str).str.zfill(5).str[:3]
df_panel_valid['zip3_year'] = df_panel_valid['zip3'] + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['zip3_year'] = df_panel_valid['zip3_year'].astype('category')

# City x Year
df_panel_valid['city_year'] = df_panel_valid['mapped_location'].astype(str) + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['city_year'] = df_panel_valid['city_year'].astype('category')

panel_data = df_panel_valid.set_index(['clinic_id', 'year'])

fe_zip5 = panel_data[['zip5_year']]
fe_zip3 = panel_data[['zip3_year']]
fe_city = panel_data[['city_year']]

outcomes_to_test = [
    'dynamic_rating',
    'current_bad_review_pct',
    'compare_ratio',
    'price_issue_ratio',
    'service_issue_ratio',
    'cosmetic_ratio'
]

for y_var in outcomes_to_test:
    print("\n\n" + "-"*80)
    print(f"DEPENDENT VARIABLE: {y_var}")
    print("-"*80)

    # Log Density
    print("Log Density")
    exog_density = sm.add_constant(panel_data[['log_density_25pct_total']])

    # City x Year
    mod_city_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_city, drop_absorbed=True)
    res_city_d = mod_city_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_city_d.summary.tables[1])

    # ZIP3 x Year
    mod_zip3_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True)
    res_zip3_d = mod_zip3_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_zip3_d.summary.tables[1])

    # ZIP5 x Year
    mod_zip5_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True)
    res_zip5_d = mod_zip5_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_zip5_d.summary.tables[1])


    # Log Entry Shock
    print("Log Entry Shock")
    exog_shock = sm.add_constant(panel_data[['log_lag_entry_shock_25pct']])

    # City x Year
    mod_city_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_city, drop_absorbed=True)
    res_city_s = mod_city_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_city_s.summary.tables[1])

    # ZIP3 x Year
    mod_zip3_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True)
    res_zip3_s = mod_zip3_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_zip3_s.summary.tables[1])

    # ZIP5 x Year
    mod_zip5_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True)
    res_zip5_s = mod_zip5_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_zip5_s.summary.tables[1])

In [ ]:
# 25%


# ZIP5 x Year
df_panel_valid['zip5_year'] = df_panel_valid['zip_str'].astype(str) + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['zip5_year'] = df_panel_valid['zip5_year'].astype('category')

# ZIP3 x Year
df_panel_valid['zip3'] = df_panel_valid['zip_str'].astype(str).str.zfill(5).str[:3]
df_panel_valid['zip3_year'] = df_panel_valid['zip3'] + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['zip3_year'] = df_panel_valid['zip3_year'].astype('category')

# City x Year
df_panel_valid['city_year'] = df_panel_valid['mapped_location'].astype(str) + "_" + df_panel_valid['year'].astype(str)
df_panel_valid['city_year'] = df_panel_valid['city_year'].astype('category')

panel_data = df_panel_valid.set_index(['clinic_id', 'year'])

fe_zip5 = panel_data[['zip5_year']]
fe_zip3 = panel_data[['zip3_year']]
fe_city = panel_data[['city_year']]

outcomes_to_test = [
    'dynamic_rating',
    'current_bad_review_pct',
    'compare_ratio',
    'price_issue_ratio',
    'service_issue_ratio',
    'cosmetic_ratio'
]

for y_var in outcomes_to_test:
    print("\n\n" + "-"*80)
    print(f"DEPENDENT VARIABLE: {y_var}")
    print("-"*80)

    # Log Density
    print("Log Density")
    exog_density = sm.add_constant(panel_data[['log_density_50pct_total']])

    # City x Year
    mod_city_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_city, drop_absorbed=True)
    res_city_d = mod_city_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_city_d.summary.tables[1])

    # ZIP3 x Year
    mod_zip3_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True)
    res_zip3_d = mod_zip3_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_zip3_d.summary.tables[1])

    # ZIP5 x Year
    mod_zip5_d = PanelOLS(panel_data[y_var], exog_density, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True)
    res_zip5_d = mod_zip5_d.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_zip5_d.summary.tables[1])


    # Log Entry Shock
    print("Log Entry Shock")
    exog_shock = sm.add_constant(panel_data[['log_lag_entry_shock_50pct']])

    # City x Year
    mod_city_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_city, drop_absorbed=True)
    res_city_s = mod_city_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_city_s.summary.tables[1])

    # ZIP3 x Year
    mod_zip3_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True)
    res_zip3_s = mod_zip3_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_zip3_s.summary.tables[1])

    # ZIP5 x Year
    mod_zip5_s = PanelOLS(panel_data[y_var], exog_shock, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True)
    res_zip5_s = mod_zip5_s.fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_zip5_s.summary.tables[1])

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

cols_to_drop = [c for c in df_panel_valid.columns if c.startswith('T_') or c in [
    'Post_Shock', 'first_treat_year', 'rel_time', 'rel_time_binned', 'treatment_event',
    'zip5_year', 'zip3_year', 'city_year'
]]
df_panel_valid = df_panel_valid.drop(columns=cols_to_drop, errors='ignore')

# first dose t=0
df_panel_valid['treatment_event'] = (df_panel_valid['lag_entry_shock_25pct_count'] > 0).astype(int)

first_treat = df_panel_valid[df_panel_valid['treatment_event'] == 1].groupby('clinic_id')['year'].min().reset_index()
first_treat.rename(columns={'year': 'first_treat_year'}, inplace=True)

df_panel_valid = df_panel_valid.merge(first_treat, on='clinic_id', how='left')

# filter out before 2014
df_panel_valid.loc[df_panel_valid['first_treat_year'] < 2014, 'first_treat_year'] = np.nan

df_panel_valid['rel_time'] = df_panel_valid['year'] - df_panel_valid['first_treat_year']

# post shock
df_panel_valid['Post_Shock'] = (df_panel_valid['rel_time'] >= 0).astype(int)
df_panel_valid.loc[df_panel_valid['first_treat_year'].isna(), 'Post_Shock'] = 0

# relative time
df_panel_valid['rel_time_binned'] = df_panel_valid['rel_time'].clip(lower=-3, upper=3)

dummies = pd.get_dummies(df_panel_valid['rel_time_binned'], prefix='T').astype(int)
dummies.columns = [c.replace('.0', '').replace('-', 'm') for c in dummies.columns]

if 'T_m1' in dummies.columns:
    dummies = dummies.drop(columns=['T_m1'])

df_panel_valid = pd.concat([df_panel_valid, dummies], axis=1)

dummy_cols = [c for c in dummies.columns if c.startswith('T_')]
for col in dummy_cols:
    df_panel_valid[col] = df_panel_valid[col].fillna(0)

# 3 level graphic size
df_panel_valid['zip5_year'] = df_panel_valid['zip_str'].astype(str) + "_" + df_panel_valid['year'].astype(str)

df_panel_valid['zip3'] = df_panel_valid['zip_str'].astype(str).str.zfill(5).str[:3]
df_panel_valid['zip3_year'] = df_panel_valid['zip3'] + "_" + df_panel_valid['year'].astype(str)

df_panel_valid['city_year'] = df_panel_valid['mapped_location'].astype(str) + "_" + df_panel_valid['year'].astype(str)

panel_data = df_panel_valid.set_index(['clinic_id', 'year'])

fe_zip5 = pd.DataFrame(panel_data['zip5_year'].astype('category').cat.codes, index=panel_data.index, columns=['zip5_year'])
fe_zip3 = pd.DataFrame(panel_data['zip3_year'].astype('category').cat.codes, index=panel_data.index, columns=['zip3_year'])
fe_city = pd.DataFrame(panel_data['city_year'].astype('category').cat.codes, index=panel_data.index, columns=['city_year'])

# regression
outcomes_to_test = [
    'dynamic_rating',
    'current_bad_review_pct',
    'price_issue_ratio',
    'compare_ratio',
    'service_issue_ratio'
]

expected_dummies = ['T_m3', 'T_m2', 'T_0', 'T_1', 'T_2', 'T_3']
exog_event_vars = [col for col in expected_dummies if col in panel_data.columns]
exog_event_data = panel_data.loc[:, ~panel_data.columns.duplicated()][exog_event_vars].astype(float)
exog_did_data = panel_data.loc[:, ~panel_data.columns.duplicated()][['Post_Shock']].astype(float)

print("Staggered DiD")
for y_var in outcomes_to_test:
    print("\n\n" + "-"*80)
    print(f"DEPENDENT VARIABLE: {y_var}")
    print("-"*80)

    exog_event = sm.add_constant(exog_event_data)

    res_es_city = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_city, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_es_city.summary.tables[1])

    res_es_zip3 = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_es_zip3.summary.tables[1])

    res_es_zip5 = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_es_zip5.summary.tables[1])

In [ ]:
cols_to_drop = [c for c in df_panel_valid.columns if c.startswith('T_') or c in [
    'Post_Shock', 'first_treat_year', 'rel_time', 'rel_time_binned', 'treatment_event',
    'zip5_year', 'zip3_year', 'city_year'
]]
df_panel_valid = df_panel_valid.drop(columns=cols_to_drop, errors='ignore')

# first dose t=0
df_panel_valid['treatment_event'] = (df_panel_valid['lag_entry_shock_50pct_count'] > 0).astype(int)

first_treat = df_panel_valid[df_panel_valid['treatment_event'] == 1].groupby('clinic_id')['year'].min().reset_index()
first_treat.rename(columns={'year': 'first_treat_year'}, inplace=True)

df_panel_valid = df_panel_valid.merge(first_treat, on='clinic_id', how='left')

# filter out before 2014
df_panel_valid.loc[df_panel_valid['first_treat_year'] < 2014, 'first_treat_year'] = np.nan

df_panel_valid['rel_time'] = df_panel_valid['year'] - df_panel_valid['first_treat_year']

# post shock
df_panel_valid['Post_Shock'] = (df_panel_valid['rel_time'] >= 0).astype(int)
df_panel_valid.loc[df_panel_valid['first_treat_year'].isna(), 'Post_Shock'] = 0

# relative time
df_panel_valid['rel_time_binned'] = df_panel_valid['rel_time'].clip(lower=-3, upper=3)

dummies = pd.get_dummies(df_panel_valid['rel_time_binned'], prefix='T').astype(int)
dummies.columns = [c.replace('.0', '').replace('-', 'm') for c in dummies.columns]

if 'T_m1' in dummies.columns:
    dummies = dummies.drop(columns=['T_m1'])

df_panel_valid = pd.concat([df_panel_valid, dummies], axis=1)

dummy_cols = [c for c in dummies.columns if c.startswith('T_')]
for col in dummy_cols:
    df_panel_valid[col] = df_panel_valid[col].fillna(0)

# 3 level graphic size
df_panel_valid['zip5_year'] = df_panel_valid['zip_str'].astype(str) + "_" + df_panel_valid['year'].astype(str)

df_panel_valid['zip3'] = df_panel_valid['zip_str'].astype(str).str.zfill(5).str[:3]
df_panel_valid['zip3_year'] = df_panel_valid['zip3'] + "_" + df_panel_valid['year'].astype(str)

df_panel_valid['city_year'] = df_panel_valid['mapped_location'].astype(str) + "_" + df_panel_valid['year'].astype(str)

panel_data = df_panel_valid.set_index(['clinic_id', 'year'])

fe_zip5 = pd.DataFrame(panel_data['zip5_year'].astype('category').cat.codes, index=panel_data.index, columns=['zip5_year'])
fe_zip3 = pd.DataFrame(panel_data['zip3_year'].astype('category').cat.codes, index=panel_data.index, columns=['zip3_year'])
fe_city = pd.DataFrame(panel_data['city_year'].astype('category').cat.codes, index=panel_data.index, columns=['city_year'])

# regression
outcomes_to_test = [
    'dynamic_rating',
    'current_bad_review_pct',
    'price_issue_ratio',
    'compare_ratio',
    'service_issue_ratio'
]

expected_dummies = ['T_m3', 'T_m2', 'T_0', 'T_1', 'T_2', 'T_3']
exog_event_vars = [col for col in expected_dummies if col in panel_data.columns]
exog_event_data = panel_data.loc[:, ~panel_data.columns.duplicated()][exog_event_vars].astype(float)
exog_did_data = panel_data.loc[:, ~panel_data.columns.duplicated()][['Post_Shock']].astype(float)

print("Staggered DiD")
for y_var in outcomes_to_test:
    print("\n\n" + "-"*80)
    print(f"DEPENDENT VARIABLE: {y_var}")
    print("-"*80)

    exog_event = sm.add_constant(exog_event_data)

    res_es_city = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_city, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nCity x Year")
    print(res_es_city.summary.tables[1])

    res_es_zip3 = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_zip3, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP3 x Year")
    print(res_es_zip3.summary.tables[1])

    res_es_zip5 = PanelOLS(panel_data[y_var], exog_event, entity_effects=True, other_effects=fe_zip5, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
    print("\nZIP5 x Year")
    print(res_es_zip5.summary.tables[1])